# Regresión lineal y referencia histórica — junio y resumen de validación
Lourdes Lema · Proyecto de titulación UDLA · Bloque 3 · v01

**Objetivo:** reajustar la regresión lineal con los cuatro primeros cortes y evaluar junio de 2011. Incorporar las métricas de marzo verificadas en Colab y calcular el promedio del MAE con igual peso por corte. No se entrena Random Forest ni se lee el archivo de prueba de septiembre.

**Instrucciones:** guardar una copia en Drive, leer cada bloque y ejecutar en orden. La primera celda solicita dos ZIP, uno después del otro:
- `Evidencias_cortes_v06(1).zip` (también admite otro nombre).
- `Evidencias_RL_H_marzo_v01.zip`, generado en tu ejecución anterior y verificado.

No descomprimir. La última celda descarga `Evidencias_RL_H_junio_v01.zip`; compartirlo para cotejar la reproducción. La verificación previa fue local y no corresponde a una nueva ejecución en Colab. Esta copia pública no conserva salidas guardadas.

## Decisiones que permanecen
Cinco predictores, H como referencia fuera de X, mínimos cuadrados con intercepto, peso igual por fila, sin imputación, escalado, transformación ni eliminación de extremos. Se conservan objetivos y predicciones negativos. No se ajusta la especificación después de observar marzo. La regresión se vuelve a entrenar desde cero con información acumulada; no se agregan manualmente correcciones a sus predicciones.

Marzo pasa a entrenamiento para pronosticar junio porque su horizonte objetivo termina el 30/05/2011, excluyendo ese instante. Sus ventas posteriores ya están observadas antes del corte de junio. Esto no modifica las predicciones ni los resultados que obtuvo marzo en el bloque anterior.


## Carga y procedencia
**Propósito:** conservar la continuidad con los cortes y resultados verificados.

**Operaciones:** cargar dos ZIP, comprobar huellas de sus CSV y recuperar entrenamiento inicial, marzo y junio. Leer las métricas previas de marzo sin volver a ajustar ese modelo. No abrir `prueba_reservada_v06.csv`.

**Resultado esperado:** 8.939 filas iniciales, 3.338 de marzo, 2.658 de junio y dos filas de métricas de marzo.

**Interpretación:** las huellas identifican los archivos auditados; diferencias detienen la ejecución para revisar su origen, no para aceptar otra versión silenciosamente.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import io, json, hashlib, platform, zipfile
import pandas as pd
import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RUTA_CORTES = None  # En Colab dejar None.
RUTA_MARZO = None   # En Colab dejar None.
def solicitar_zip(ruta_local, mensaje):
    if ruta_local is not None:
        return Path(ruta_local)
    from google.colab import files
    print(mensaje)
    subidos = files.upload()
    nombres = [n for n in subidos if n.lower().endswith('.zip')]
    if len(nombres) != 1:
        raise ValueError('Seleccione un solo ZIP en cada solicitud.')
    return Path(nombres[0])
def leer_miembro(z, nombre):
    encontrados = [n for n in z.namelist() if Path(n).name == nombre]
    assert len(encontrados) == 1, f'Archivo ausente o ambiguo: {nombre}'
    return z.read(encontrados[0])
ruta_cortes = solicitar_zip(RUTA_CORTES, 'Primero: cargue Evidencias_cortes_v06(1).zip')
HUELLAS_CORTES = {'entrenamiento_inicial_v06.csv': '65d7dc09069779841830e0d596ecd4eb6cd78a61f193b12d8a7ffc08a356cd59', 'validacion_v06.csv': 'bfb661e6f35357af531beebfba7f6bf0acffeb126a0a259c129ad4479a8bcedc'}
contenido = {}
with zipfile.ZipFile(ruta_cortes) as z:
    for nombre, huella in HUELLAS_CORTES.items():
        contenido[nombre] = leer_miembro(z, nombre)
        assert hashlib.sha256(contenido[nombre]).hexdigest() == huella, f'Versión distinta: {nombre}'
inicial = pd.read_csv(io.BytesIO(contenido['entrenamiento_inicial_v06.csv']), dtype={'Customer ID':'string'})
validacion = pd.read_csv(io.BytesIO(contenido['validacion_v06.csv']), dtype={'Customer ID':'string'})
ruta_marzo = solicitar_zip(RUTA_MARZO, 'Segundo: cargue Evidencias_RL_H_marzo_v01.zip')
HUELLA_METRICAS_MARZO = '72c60cf9f766ede5e027daa7c9d17cf22b7e4ab1ab61eaebfe671add27a244fe'
with zipfile.ZipFile(ruta_marzo) as z:
    bytes_metricas = leer_miembro(z, 'metricas_marzo.csv')
    assert hashlib.sha256(bytes_metricas).hexdigest() == HUELLA_METRICAS_MARZO, 'Métricas de marzo distintas de las verificadas.'
    bitacora_marzo = json.loads(leer_miembro(z, 'bitacora.json'))
metricas_marzo = pd.read_csv(io.BytesIO(bytes_metricas))
assert bitacora_marzo['sha256_csv_entrada'] == HUELLAS_CORTES
print('Inicial:',len(inicial),'| Validación disponible:',len(validacion))
print('Métricas de marzo recuperadas:',len(metricas_marzo))
print('scikit-learn:',sklearn.__version__)


## Partición acumulativa y controles
**Propósito:** formar el conjunto que estaría disponible el 01/06/2011.

**Operaciones:** restaurar tipos, validar claves, fórmulas y conteos; unir los tres cortes iniciales con marzo y reservar junio para evaluación. Verificar que todas las etiquetas de entrenamiento terminen antes o en la fecha simulada.

**Resultado esperado:** 12.277 observaciones para ajustar y 2.658 para evaluar, con exactamente cinco predictores.

**Interpretación:** los clientes pueden repetirse entre cortes. No son observaciones independientes; se mantienen porque representan pronósticos en fechas distintas. No se realizan pruebas de significación convencional ni se supone que el mayor volumen de entrenamiento garantice una mejora.


In [ ]:
predictores = ['recencia_dias','frecuencia_180d','diversidad_180d',
               'importe_neto_180d','importe_neto_por_compra_180d']
objetivo = 'objetivo_neto_90d'
referencia = 'referencia_neto_90d'
variables = predictores + [referencia, objetivo]
controles = []
def verificar(nombre, condicion):
    controles.append({'control':nombre,'aprobado':bool(condicion)})
    if not condicion: raise AssertionError(nombre)
for nombre,d in [('inicial',inicial),('validacion',validacion)]:
    verificar(nombre+' columnas',set(d.columns)==set(['Customer ID','fecha_corte']+variables))
    d['Customer ID'] = d['Customer ID'].astype('string')
    d['fecha_corte'] = pd.to_datetime(d['fecha_corte'],errors='raise')
    for c in variables: d[c] = pd.to_numeric(d[c],errors='raise')
    verificar(nombre+' sin nulos',not d.isna().any().any())
    verificar(nombre+' claves únicas',not d.duplicated(['Customer ID','fecha_corte']).any())
    verificar(nombre+' finitos',np.isfinite(d[variables]).all().all())
    verificar(nombre+' compras positivas',d.frecuencia_180d.ge(1).all())
    verificar(nombre+' P=M/F',np.allclose(d.importe_neto_por_compra_180d,d.importe_neto_180d/d.frecuencia_180d,rtol=1e-12,atol=1e-9))
cortes_iniciales = pd.to_datetime(['2010-06-01','2010-09-01','2010-12-01'])
verificar('Conteos iniciales',inicial.groupby('fecha_corte').size().to_dict()==dict(zip(cortes_iniciales,[2652,2822,3465])))
verificar('Conteos validacion',validacion.groupby('fecha_corte').size().to_dict()=={
    pd.Timestamp('2011-03-01'):3338,pd.Timestamp('2011-06-01'):2658})
marzo = validacion.loc[validacion.fecha_corte.eq('2011-03-01')].copy()
junio = validacion.loc[validacion.fecha_corte.eq('2011-06-01')].copy()
train = pd.concat([inicial,marzo],ignore_index=True)
for d in [train,junio]:
    d.sort_values(['fecha_corte','Customer ID'],inplace=True)
    d.reset_index(drop=True,inplace=True)
verificar('12277 filas entrenamiento',len(train)==12277)
verificar('2658 filas evaluación',len(junio)==2658)
verificar('Clave acumulada única',not train.duplicated(['Customer ID','fecha_corte']).any())
verificar('Junio fuera del entrenamiento',train.fecha_corte.lt(pd.Timestamp('2011-06-01')).all())
verificar('Etiquetas disponibles',((train.fecha_corte+pd.Timedelta(days=90))<=pd.Timestamp('2011-06-01')).all())
verificar('Métricas de marzo válidas',len(metricas_marzo)==2 and set(metricas_marzo.modelo)=={'Referencia H','Regresión lineal'} and metricas_marzo.corte.eq('2011-03-01').all() and metricas_marzo.n.eq(3338).all())
verificar('Métricas de marzo finitas',np.isfinite(metricas_marzo[['MAE_GBP','RMSE_GBP','R2']]).all().all())
X_train,y_train = train[predictores].copy(),train[objetivo].copy()
X_junio,y_junio = junio[predictores].copy(),junio[objetivo].copy()
print('X entrenamiento:',X_train.shape,'| X junio:',X_junio.shape)
print('Último fin exclusivo de etiquetas:',(train.fecha_corte+pd.Timedelta(days=90)).max())
print('Controles aprobados:',len(controles))


## Reajuste de regresión lineal
**Propósito:** estimar de nuevo los coeficientes con toda la información disponible antes de junio.

**Operaciones:** crear un modelo nuevo con la misma especificación del bloque anterior; ejecutar `fit` sobre los cuatro cortes y `predict` sobre junio. Tomar H de junio como predicción histórica y verificar la ecuación lineal.

**Resultado esperado:** 2.658 predicciones por alternativa y seis parámetros de regresión (intercepto y cinco coeficientes).

**Interpretación:** cambiarán los coeficientes porque cambió el entrenamiento, no porque se hayan optimizado usando junio. Sus signos no son efectos causales ni sus magnitudes son una clasificación de importancia. Se preservan todas las predicciones negativas.


In [ ]:
pred_h = junio[referencia].to_numpy(copy=True)
modelo = LinearRegression(fit_intercept=True, copy_X=True, positive=False)
modelo.fit(X_train, y_train)
pred_rl = modelo.predict(X_junio)
verificar('Predicciones finitas',np.isfinite(pred_rl).all() and np.isfinite(pred_h).all())
verificar('Predicción coincide con ecuación',np.allclose(pred_rl,
          X_junio.to_numpy()@modelo.coef_+modelo.intercept_,rtol=1e-12,atol=1e-8))
coeficientes = pd.DataFrame({'termino':['intercepto']+predictores,
                           'coeficiente':[float(modelo.intercept_)]+modelo.coef_.tolist()})
print(coeficientes.to_string(index=False))
print('Rango de la matriz centrada:',modelo.rank_,'de',len(predictores))
print('Predicciones negativas RL conservadas:',int((pred_rl<0).sum()))


## Evaluación de junio
**Propósito:** medir el desempeño en el segundo corte de validación bajo el mismo protocolo.

**Operaciones:** calcular MAE, RMSE y R² en los mismos clientes para H y regresión lineal; comprobar MAE y RMSE contra sus fórmulas y exportar errores individuales.

**Resultado esperado:** dos filas de métricas de junio, sin redondear los valores almacenados.

**Interpretación:** MAE y RMSE están en GBP por horizonte de 90 días; menor es mejor. R² no es exactitud y puede ser negativo. Las diferencias entre marzo y junio pueden depender de las cohortes y los períodos; no prueban estacionalidad. Este bloque no evalúa la priorización comercial.


In [ ]:
filas=[]
for nombre,pred in [('Referencia H',pred_h),('Regresión lineal',pred_rl)]:
    mae=float(mean_absolute_error(y_junio,pred))
    rmse=float(np.sqrt(mean_squared_error(y_junio,pred)))
    r2=float(r2_score(y_junio,pred))
    error=y_junio.to_numpy()-pred
    verificar(nombre+' MAE comprobado',np.isclose(mae,np.mean(np.abs(error)),rtol=1e-12))
    verificar(nombre+' RMSE comprobado',np.isclose(rmse,np.sqrt(np.mean(error**2)),rtol=1e-12))
    filas.append({'modelo':nombre,'corte':'2011-06-01','n':len(junio),'MAE_GBP':mae,'RMSE_GBP':rmse,'R2':r2})
metricas=pd.DataFrame(filas)
print(metricas.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
delta=float(metricas.iloc[1].MAE_GBP-metricas.iloc[0].MAE_GBP)
print(f'Diferencia MAE RL menos H: {delta:.6f} GBP')
if metricas.iloc[0].MAE_GBP>0:
    print(f'Reducción relativa del MAE frente a H: {-100*delta/metricas.iloc[0].MAE_GBP:.4f}%')
predicciones=junio[['Customer ID','fecha_corte',objetivo,referencia]].copy()
predicciones['prediccion_RL']=pred_rl
predicciones['error_RL_observado_menos_predicho']=y_junio.to_numpy()-pred_rl
predicciones['error_H_observado_menos_predicho']=y_junio.to_numpy()-pred_h
print('Evaluación de junio completada. No se selecciona todavía modelo definitivo.')


## Resumen con igual peso por corte
**Propósito:** aplicar el criterio de comparación acordado antes de consultar la prueba final.

**Operaciones:** unir las métricas verificadas de marzo con las nuevas de junio y calcular, para cada alternativa:

$$\overline{MAE} = \frac{MAE_{marzo}+MAE_{junio}}{2}.$$

**Resultado esperado:** una tabla con los dos MAE por alternativa y su promedio simple. Se incluye la diferencia promedio RL−H y su variación relativa frente a H.

**Interpretación:** cada corte aporta 50 %, aunque marzo tenga más clientes. No se agrupan todos los errores en una sola media, pues eso daría mayor peso a marzo. El resultado describe únicamente las dos alternativas evaluadas; faltan las configuraciones de Random Forest para cerrar la selección. No se cambia la métrica principal según cuál favorezca a un modelo.


In [ ]:
metricas_junio = metricas.copy()
metricas_por_corte = pd.concat([metricas_marzo,metricas_junio],ignore_index=True)
verificar('Dos modelos y dos cortes',len(metricas_por_corte)==4 and not metricas_por_corte.duplicated(['modelo','corte']).any())
resumen = metricas_por_corte.pivot(index='modelo',columns='corte',values='MAE_GBP').rename(
    columns={'2011-03-01':'MAE_marzo_GBP','2011-06-01':'MAE_junio_GBP'})
resumen['MAE_promedio_GBP'] = (resumen.MAE_marzo_GBP+resumen.MAE_junio_GBP)/2
verificar('Promedio explícito coincide',np.allclose(resumen.MAE_promedio_GBP,
    metricas_por_corte.groupby('modelo').MAE_GBP.mean().reindex(resumen.index),rtol=1e-12))
resumen = resumen.reset_index()
resumen.columns.name = None
print(resumen.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
h=float(resumen.loc[resumen.modelo.eq('Referencia H'),'MAE_promedio_GBP'].iloc[0])
rl=float(resumen.loc[resumen.modelo.eq('Regresión lineal'),'MAE_promedio_GBP'].iloc[0])
print(f'Diferencia del MAE promedio RL menos H: {rl-h:.6f} GBP')
if h>0: print(f'Variación relativa RL frente a H: {100*(rl-h)/h:.4f}% (positiva indica más error)')
print('Validación de RL y H completada; selección global pendiente de Random Forest.')


## Evidencias y bitácora
**Propósito:** conservar resultados reproducibles y continuidad con la ejecución anterior.

**Operaciones:** exportar tablas de junio, métricas de ambos cortes, resumen, coeficientes, controles y datos utilizados; registrar versiones, huellas y parámetros. Descargar un ZIP.

**Resultado esperado:** `Evidencias_RL_H_junio_v01.zip`.

**Interpretación:** compartir el ZIP para verificar reproducción. Conservar también este notebook en Drive para integrarlo posteriormente al repositorio GitHub. Las huellas de origen de marzo se mantienen; no se sustituye ni se modifica su evidencia original.


In [ ]:
salida=Path('salidas_RL_H_junio_v01');salida.mkdir(exist_ok=True)
tablas={'metricas_junio.csv':metricas_junio,'metricas_por_corte.csv':metricas_por_corte,
 'resumen_validacion.csv':resumen,'predicciones_junio.csv':predicciones,
 'coeficientes_RL_junio.csv':coeficientes,'controles.csv':pd.DataFrame(controles),
 'entrenamiento_utilizado.csv':train,'validacion_junio_utilizada.csv':junio}
archivos=[]
for nombre,tabla in tablas.items():
    p=salida/nombre;tabla.to_csv(p,index=False);archivos.append(p)
bitacora={'version':'bloque3_v01','fecha_utc':datetime.now(timezone.utc).isoformat(),
 'sha256_zip_cortes':hashlib.sha256(ruta_cortes.read_bytes()).hexdigest(),
 'sha256_zip_marzo':hashlib.sha256(ruta_marzo.read_bytes()).hexdigest(),
 'sha256_csv_cortes':HUELLAS_CORTES,'sha256_metricas_marzo':HUELLA_METRICAS_MARZO,
 'predictores':predictores,'objetivo':objetivo,'referencia':referencia,
 'parametros_RL':modelo.get_params(),'rango_X_centrada':int(modelo.rank_),
 'filas_entrenamiento':len(train),'filas_validacion':len(junio),
 'versiones':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scikit-learn':sklearn.__version__},
 'reglas':'Mínimos cuadrados con intercepto; peso igual por fila; sin escalado, imputación, filtros, truncamiento o cambio de predictores.',
 'agregacion':'MAE de marzo y junio con igual peso 0.5; no promedio agrupado por observaciones.',
 'alcance':'Reajuste y validación junio; métricas marzo reutilizadas sin reentrenar; prueba final no leída.',
 'pendientes':['Reproducción de este bloque por Lourdes y cotejo','Configuraciones Random Forest','Selección y prueba final','Priorización comercial'],
 'sha256_salidas':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in archivos}}
p=salida/'bitacora.json';p.write_text(json.dumps(bitacora,ensure_ascii=False,indent=2));archivos.append(p)
paquete=Path('Evidencias_RL_H_junio_v01.zip')
with zipfile.ZipFile(paquete,'w',zipfile.ZIP_DEFLATED) as z:
    for p in archivos:z.write(p,p.name)
print('Evidencias generadas:',paquete.name)
try:
    from google.colab import files
except ImportError:
    print('Ejecución local verificada; descarga automática solo en Colab.')
else:
    files.download(str(paquete))


## Lectura y defensa
- ¿Por qué se pueden incorporar las etiquetas de marzo para entrenar en junio?
- ¿Por qué entrenar con 12.277 filas no significa tener 12.277 clientes distintos?
- ¿Por qué usamos 50 % de peso por corte y no por cantidad de clientes?
- ¿Por qué no se puede declarar aún un ganador definitivo ni afirmar mayor venta causada por la priorización?

**Fuentes técnicas:** las mismas del bloque anterior, sin añadir entradas duplicadas a la bibliografía: scikit-learn developers (s. f.), *LinearRegression* y *Metrics and scoring: Quantifying the quality of predictions*.

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics
